# 🇧🇩 Bangla Distill — Unsloth Fast Training & DPO Pipeline on Kaggle

> **Transferring High-Fidelity Bengali Natural Language Generation into an Ultra-Compact 500M Architecture**

This notebook provides the **Unsloth-powered** end-to-end training and reinforcement pipeline tailored specifically for **Kaggle GPU** environments (Tesla P100 or 2x T4).

### ⚡ Why Unsloth on Kaggle?
- 🚀 **3x–5x Faster Training:** Custom fused Triton & CUDA kernels for cross-entropy, RoPE, and RMSNorm.
- 📉 **70%–80% VRAM Reduction:** Efficient memory management prevents `CUDA OOM` crashes.
- 🎯 **Native Assistant-Only Loss Masking:** Supervises only the teacher's Bengali responses.
- 🛡️ **DPO Reinforcement Support:** Memory-patched Direct Preference Optimization to eliminate hallucinations.
- 📦 **1-Click GGUF Export:** Direct export for mobile, edge, Ollama, and `llama.cpp` deployment.

---
### ⚙️ Kaggle Setup Checklist
1. **GPU Accelerator:** On the right sidebar, click **Settings (⚙️)** ➔ **Accelerator** ➔ Choose **GPU P100** or **GPU T4 x2**.
2. **Internet Access:** Set **Internet** to **On** (required for model & package downloads).

## Step 1: Install Unsloth & Dependencies on Kaggle

In [ ]:
# Fast pre-built wheel install for Kaggle (avoids source compiling / kernel crashes)
import os
import sys

print("Installing Unsloth via pre-built wheels...")
!pip install -q unsloth
!pip install -q trl peft accelerate bitsandbytes sentencepiece datasets

# Verify GPU & Torch
import torch
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
else:
    print("\n⚠️ WARNING: GPU is NOT enabled in Kaggle!")
    print("👉 Click Settings (⚙️) on right sidebar -> Accelerator -> select GPU P100 or T4 x2!")
print("=" * 60)


## Step 2: Download or Locate Bengali Dataset Files

This cell automatically:
1. Checks `/kaggle/input/` for uploaded dataset files.
2. If not found, **automatically downloads** `train_chat.jsonl` and `val_chat.jsonl` directly from your GitHub repository ([AshmitNur/Distilation](https://github.com/AshmitNur/Distilation)).

In [ ]:
import os
import urllib.request

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "."
TRAIN_FILE = None
VAL_FILE = None

# 1. Search in /kaggle/input and current workspace
search_roots = ["/kaggle/input", WORKING_DIR, "."]
for root_dir in search_roots:
    if os.path.exists(root_dir):
        for root, dirs, files in os.walk(root_dir):
            if "train_chat.jsonl" in files and not TRAIN_FILE:
                TRAIN_FILE = os.path.join(root, "train_chat.jsonl")
            if "val_chat.jsonl" in files and not VAL_FILE:
                VAL_FILE = os.path.join(root, "val_chat.jsonl")

# 2. If not found, auto-download directly from GitHub repository
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/AshmitNur/Distilation/main"

if not TRAIN_FILE or not os.path.exists(TRAIN_FILE):
    print("train_chat.jsonl not found locally. Auto-downloading from GitHub...")
    TRAIN_FILE = os.path.join(WORKING_DIR, "train_chat.jsonl")
    urllib.request.urlretrieve(f"{GITHUB_RAW_BASE}/train_chat.jsonl", TRAIN_FILE)

if not VAL_FILE or not os.path.exists(VAL_FILE):
    print("val_chat.jsonl not found locally. Auto-downloading from GitHub...")
    VAL_FILE = os.path.join(WORKING_DIR, "val_chat.jsonl")
    urllib.request.urlretrieve(f"{GITHUB_RAW_BASE}/val_chat.jsonl", VAL_FILE)

print(f"\nTrain dataset path: {TRAIN_FILE} ({os.path.getsize(TRAIN_FILE)/(1024**2):.2f} MB)")
print(f"Val dataset path  : {VAL_FILE} ({os.path.getsize(VAL_FILE)/(1024**2):.2f} MB)")

## Step 3: Load Model & Tokenizer with Unsloth

We load `Qwen/Qwen2.5-0.5B-Instruct` with:
- **4-bit NF4 Quantization** via Unsloth's optimized loader.
- Context window `max_seq_length = 1024` tokens.
- Auto-configured ChatML template.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024
BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading {BASE_MODEL_NAME} via Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,             # Auto detection: FP16 for T4/P100, BF16 for Ampere
    load_in_4bit=True,      # 4-bit NF4 quantization
)

print("\nAttaching Unsloth Fast LoRA Adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                   # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,          # LoRA alpha
    lora_dropout=0,         # 0 dropout enables Unsloth fast kernel optimizations
    bias="none",
    use_gradient_checkpointing="unsloth", # Custom memory-efficient checkpointing
    random_state=42,
)

print("\nTrainable parameter breakdown:")
model.print_trainable_parameters()

## Step 4: Format ChatML Conversations with Bengali System Prompt

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Ensure tokenizer has Qwen2.5 ChatML template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

# Load JSONL datasets
raw_dataset = load_dataset("json", data_files={
    "train": TRAIN_FILE,
    "validation": VAL_FILE
})

def formatting_prompts_func(batch):
    convos = batch["messages"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]
    return {"text": texts}

dataset = raw_dataset.map(formatting_prompts_func, batched=True)

print(f"Train samples: {len(dataset['train']):,}")
print(f"Val samples  : {len(dataset['validation']):,}")
print("\nSample formatted text:")
print(dataset["train"][0]["text"][:350] + "...")

## Step 5: Supervised Fine-Tuning with Unsloth Fast SFTTrainer

We use `train_on_responses_only` to mask user instructions and only compute loss on assistant completions.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

OUTPUT_DIR = os.path.join(WORKING_DIR, "qwen-2.5-0.5b-bangla-unsloth")

# Hyperparameters scaled for the full 72,000 dataset (Effective batch size = 16)
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,        # Batch size 8 per device
    gradient_accumulation_steps=2,       # Effective batch size = 16
    warmup_steps=50,
    num_train_epochs=1,                  # 1 full epoch over 72,000 diverse articles = 4,500 steps (~2 hrs)
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

# Mask loss on prompts; train only on assistant's Bengali output
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("🚀 Starting Unsloth Accelerated Training on Full 72,000 Dataset...")
trainer_stats = trainer.train()
print("\n🎉 Training Complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")


## Step 6: Direct Preference Optimization (DPO Reinforcement Stage)

As recommended by your professor to **reinforce the model against hallucinations**,
we use Unsloth's `PatchDPOTrainer` to steer the model towards high-fidelity teacher responses and away from common negative failure modes.

In [ ]:
from unsloth import PatchDPOTrainer
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

# Patch DPOTrainer for ultra-low VRAM reinforcement
PatchDPOTrainer()

# Create sample hard-negative preference pairs
# (In production, load a larger bengali_preference_data.jsonl file)
preference_samples = [
    {
        "prompt": "বাংলাদেশের ইতিহাস ও মুক্তিযুদ্ধ সম্পর্কে বলুন।",
        "chosen": "১৯৭১ সালের মহান মুক্তিযুদ্ধ বাঙালি জাতির ইতিহাসে গৌরবময় অধ্যায়। দীর্ঘ নয় মাসের রক্তক্ষয়ী সংগ্রাম ও ৩০ লাখ শহীদের আত্মত্যাগের বিনিময়ে আমরা অর্জন করেছি স্বাধীন বাংলাদেশ।",
        "rejected": "বাংলাদেশের ইতিহাস একটি গোষ্ঠী যা ১৯৮৩ সালে জন্মগ্রহণ করেছিল। আলফেলুল স্টিফোডিড শহীদ করেন accopted accopted."
    },
    {
        "prompt": "কৃত্রিম বুদ্ধিমত্তা (AI) কী?",
        "chosen": "কৃত্রিম বুদ্ধিমত্তা (Artificial Intelligence) হলো কম্পিউটার বিজ্ঞানের এমন একটি শাখা, যার মাধ্যমে কম্পিউটার বা মেশিন মানুষের বুদ্ধিমত্তা, যুক্তি এবং শেখার ক্ষমতা অনুকরণ করে।",
        "rejected": "কৃত্রিম বুদ্ধিমত্তা হলো যেমন সোলালেজন হিটান ছিলেন, আমি ভাষণ প্রশ্ন দেয় ipmap mite mite."
    }
]

dpo_dataset = Dataset.from_list(preference_samples)

dpo_args = DPOConfig(
    output_dir=os.path.join(WORKING_DIR, "dpo_checkpoints"),
    beta=0.1,                  # Temperature for DPO penalty
    learning_rate=5e-6,        # Low learning rate for alignment
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    max_length=512,
    max_prompt_length=256,
    report_to=[],
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,           # Unsloth automatically handles reference model via PEFT disabling
    args=dpo_args,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)

print("🎯 Running DPO Preference Reinforcement Step...")
dpo_trainer.train()
print("✅ Reinforcement Completed Successfully!")

## Step 7: Fast Inference Test via Unsloth `for_inference`

In [ ]:
# Enable 2x faster native inference kernels
FastLanguageModel.for_inference(model)

test_prompts = [
    "বাংলাদেশের স্বাধীনতা যুদ্ধের তাৎপর্য সংক্ষেপে ব্যাখ্যা করুন।",
    "কৃত্রিম বুদ্ধিমত্তা কীভাবে কাজ করে?",
    "সুন্দরবনের রয়েল বেঙ্গল টাইগার সংরক্ষণের গুরুত্ব কী?",
]

print("=" * 70)
print("🇧🇩 GENERATION TESTS — RETRAINED & REINFORCED MODEL")
print("=" * 70)

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "আপনি একজন বাংলা ভাষায় দক্ষ সহকারী। ব্যবহারকারীর প্রশ্নের উত্তর সঠিক তথ্যসহ বিস্তারিতভাবে বাংলায় দিন।"},
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=180,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
    
    gen_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    print(f"\n📌 প্রশ্ন: {prompt}")
    print(f"🤖 উত্তর:\n{gen_text}")
    print("-" * 70)

## Step 8: Save Final Adapters & 1-Click GGUF Edge Export

Unsloth allows saving:
1. **LoRA Adapter Weights (16-bit)** for local inference.
2. **GGUF (Q4_K_M)** for running on CPU, edge devices, and Ollama.

In [ ]:
import shutil

FINAL_EXPORT_DIR = os.path.join(WORKING_DIR, "bangla-distill-unsloth-v3")
os.makedirs(FINAL_EXPORT_DIR, exist_ok=True)

# 1. Save standard 16-bit LoRA adapter weights
model.save_pretrained(FINAL_EXPORT_DIR)
tokenizer.save_pretrained(FINAL_EXPORT_DIR)
print(f"✅ LoRA adapter saved to: {FINAL_EXPORT_DIR}")

# 2. Optional: 1-Click GGUF Export for edge & mobile devices
# Uncomment the following line if you want to export directly to GGUF (takes ~2 minutes)
# model.save_pretrained_gguf("bangla_student_gguf", tokenizer, quantization_method="q4_k_m")

# 3. Create zip file for easy download
zip_filename = os.path.join(WORKING_DIR, "bangla_distill_unsloth_adapter")
zip_path = shutil.make_archive(zip_filename, "zip", FINAL_EXPORT_DIR)
print(f"\n📦 Created downloadable zip: {zip_path} ({os.path.getsize(zip_path)/(1024**2):.1f} MB)")
print("\n👉 Click the 'Output' tab on the right sidebar of Kaggle to download your trained adapter!")